In [ ]:
from datetime import datetime
from pdb import set_trace
from time import time, sleep
from random import randint

import numpy as np
import torch as th
import torch.nn as nn

from gymnasium.spaces import Dict
from minigrid.core.world_object import Ball, Box, Key
from minigrid.wrappers import FullyObsWrapper
from gymnasium.wrappers import RecordVideo
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.utils import set_random_seed

from custom_mini_env import ObjObsWrapper, CustomShapesEnvSimpleEval
from global_utils import (SAVE_FREQUENCY, DEFAULT_SIZE, DistType)


# Transformer base feature extractor can be used with this env

class LightweightGridTransformerExtractor(BaseFeaturesExtractor):
    """
    Very lightweight transformer for fast training on symbolic MiniGrid
    """
    def __init__(self, observation_space: Dict, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        
        # Image properties
        img_space = observation_space.spaces["image"]
        img_flatten_dim = np.prod(img_space.shape)
        
        # Spec properties
        spec_space = observation_space.spaces["specs"]
        spec_dim = spec_space.shape[0]
        
        # Combined input
        total_dim = img_flatten_dim + spec_dim
        
        # Simple projection
        self.input_proj = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
        )
        
        # Tiny transformer (single layer)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        
        self.output_proj = nn.Sequential(
            nn.LayerNorm(128),
            nn.Linear(128, features_dim),
            nn.Tanh()
        )

    def forward(self, observations) -> th.Tensor:
        # Flatten image - using reshape instead of view and ensuring tensor is contiguous
        image = observations["image"].float()
        if image.dim() == 3:
            image = image.unsqueeze(0)  # Add batch dimension if needed
        image_flat = image.contiguous().reshape(image.shape[0], -1)
        
        # Combine with spec
        spec = observations["specs"].float()
        combined = th.cat([image_flat, spec], dim=1)
        
        # Project
        x = self.input_proj(combined).unsqueeze(1)  # (B, 1, 128)
        
        # Tiny transformer
        encoded = self.transformer(x)
        
        # Output
        features = encoded.squeeze(1)
        return self.output_proj(features) 


class EfficientGridTransformerExtractor(BaseFeaturesExtractor):
    """
    More efficient transformer for MiniGrid symbolic observations
    Uses attention between grid cells and specification
    """
    def __init__(self, observation_space: Dict, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        
        # MiniGrid image properties
        img_space = observation_space.spaces["image"]
        self.img_height = img_space.shape[0]
        self.img_width = img_space.shape[1]
        self.img_channels = img_space.shape[2]
        self.num_cells = self.img_height * self.img_width
        
        # Simple grid cell encoder
        self.cell_encoder = nn.Sequential(
            nn.Linear(self.img_channels, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
        )
        
        # Spec processor
        spec_space = observation_space.spaces["specs"]
        self.spec_dim = spec_space.shape[0]
        self.spec_encoder = nn.Sequential(
            nn.Linear(self.spec_dim, 128),
            nn.ReLU(),
        )
        
        # Cross-attention: let spec attend to relevant grid cells
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=128,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )
        
        # Self-attention for grid cells
        self.grid_self_attention = nn.MultiheadAttention(
            embed_dim=128,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )
        
        # Output processing
        self.output_net = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, features_dim),
            nn.Tanh()
        )
        
        # Normalization layers
        self.norm1 = nn.LayerNorm(128)
        self.norm2 = nn.LayerNorm(128)

    def forward(self, observations) -> th.Tensor:
        batch_size = observations["image"].shape[0]
        
        # Encode grid cells
        grid_image = observations["image"].float()

        # Ensure batch dim
        if grid_image.dim() == 3:
            grid_image = grid_image.unsqueeze(0)

        # If channels-first (B, C, H, W) -> convert to channels-last (B, H, W, C)
        if grid_image.shape[1] == self.img_channels and grid_image.dim() == 4:
            # assume (B, C, H, W)
            grid_image = grid_image.permute(0, 2, 3, 1)

        # Make contiguous and reshape safely
        grid_image = grid_image.contiguous()
        grid_flat = grid_image.reshape(batch_size, -1, self.img_channels)  # (B, num_cells, C)
        grid_encoded = self.cell_encoder(grid_flat)  # (B, num_cells, 128)
        
        # Encode spec
        spec = observations["specs"].float()
        spec_encoded = self.spec_encoder(spec).unsqueeze(1)  # (B, 1, 128)
        
        # Self-attention on grid to capture spatial relationships
        grid_attended, _ = self.grid_self_attention(
            query=grid_encoded, key=grid_encoded, value=grid_encoded
        )
        grid_encoded = self.norm1(grid_encoded + grid_attended)
        
        # Cross-attention: spec queries grid keys/values
        # This lets the spec "look for" relevant objects in the grid
        spec_attended, attention_weights = self.cross_attention(
            query=spec_encoded,
            key=grid_encoded,
            value=grid_encoded
        )
        
        # Combine and project
        final_features = self.norm2(spec_encoded + spec_attended)
        final_features = final_features.squeeze(1)  # (B, 128)
        
        return self.output_net(final_features)
    
    
def make_env(seed=0, size=9, num_objects=2, max_steps=200, record=False, video_dir="videos"
            ,target_type=Ball,target_color='blue',episode_trig=5
            ,render_mode="rgb_array", is_partial=True, dist_type=DistType.NO_DIST):
    
    def _thunk():
        env = CustomShapesEnvSimpleEval(size=size, num_objects=num_objects, max_steps=max_steps, render_mode=render_mode
                    ,target_type=target_type,target_color=target_color,dist_type=dist_type)

        if not is_partial:	env = FullyObsWrapper(env)

        env = ObjObsWrapper(env)
        
        if record:
            Path(video_dir).mkdir(parents=True, exist_ok=True)
            env = RecordVideo(env, video_folder=video_dir, episode_trigger=lambda ep: (ep % episode_trig) == 0)
            
        env.reset(seed=seed)
        
        return env
    
    return _thunk    


def main(train=True, render=True, model_ref="", num_distractors=0,size=DEFAULT_SIZE,obj_type=None
         ,obj_color=None,existing=False,dist_type=DistType.NO_DIST,seed=42):

    # policy_kwargs = dict(
    #     features_extractor_class=EfficientGridTransformerExtractor,
    #     features_extractor_kwargs=dict(features_dim=256),
    #     net_arch=[128, 64],  # Policy head
    #     activation_fn=nn.ReLU,
    # )
    
    # policy_kwargs = dict(
    #     features_extractor_class=MiniGridTransformerExtractor,
    #     features_extractor_kwargs=dict(
    #         features_dim=256,
    #         d_model=128,
    #         nhead=8,
    #         num_layers=3
    #     ),
    #     net_arch=[64, 32]
    # )
    
    policy_kwargs = dict(
        features_extractor_class=LightweightGridTransformerExtractor,
        features_extractor_kwargs=dict(features_dim=256),
        net_arch=[64, 32]
    )
    
    
    stamp = datetime.fromtimestamp(time()).strftime("%Y%m%d-%H%M%S")
    steps = None

    if train:
        env = make_env(
            seed=seed,
            size=size,
            num_objects=1+num_distractors,
            max_steps=steps,
            target_type=obj_type,
            target_color=obj_color,
            dist_type=dist_type,
            is_partial=True)()

        checkpoint_callback = CheckpointCallback(
            save_freq=SAVE_FREQUENCY,
            save_path=f"./models/ppo/simple/minigrid_trans_{num_distractors}_{dist_type.name}_{seed}_{stamp}/",
            name_prefix=f"trans_baseline_{num_distractors}_{size}",
        )
        
        if existing: # finetuning an existing model
            ppo = PPO("MultiInputPolicy",
                    env,
                    policy_kwargs=policy_kwargs,
                    verbose=1,
                    ent_coef=0.02)
            model = ppo.load(f"{model_ref}", env=env, tensorboard_log=f"./logs/ppo/simple/minigrid_trans_{num_distractors}_{size}_{dist_type.name}_{seed}_tensorboard/")
        else:
            model = PPO(
				"MultiInputPolicy",
				env,
				policy_kwargs=policy_kwargs,
				verbose=1,
				tensorboard_log=f"./logs/ppo/simple/minigrid_trans_{num_distractors}_{size}_{dist_type.name}_{seed}_tensorboard/",
			)

        model.learn(
            total_timesteps=TOTAL_TIMESTEPS,
            tb_log_name=f"{stamp}",
            callback=checkpoint_callback,
        )
    else: print("FOR EVALUATION, PLEASE USE THE mlp_baseline.ipynb script")



if __name__ == "__main__":
    # Set random seeds for reproducibility
    SEED = 50
    TOTAL_TIMESTEPS = 1000_000
    set_random_seed(SEED)
    
    print(TOTAL_TIMESTEPS, SEED, DEFAULT_SIZE)
    
    for i in range(1,4):
        
        main(seed=SEED,
        train=False,
        render=True,
        obj_color="yellow",
        obj_type=Key,
        dist_type=DistType(i),
        model_ref="models/ppo/simple/minigrid_trans_0_BASIC_SKILLS_42_20251115-192757/trans_baseline_0_8_500000_steps.zip",
        num_distractors=0)
        
        sleep(1)


1000000 50 8
FOR EVALUATION, PLEASE USE THE mlp_baseline.ipynb script
